In [ ]:
from pyspark.sql.functions import col
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = spark.read.parquet("data/spark-output/cleaned_reviews")


In [ ]:
df.printSchema()
df.count()


In [ ]:
from pyspark.sql.functions import year, month, count

monthly_timeline = (
    df
    .withColumn("year", year("review_date"))
    .withColumn("month", month("review_date"))
    .groupBy("year", "month")
    .agg(count("*").alias("n_reviews"))
    .orderBy("year", "month")
)

monthly_timeline.show(10)


In [ ]:
timeline_pd = monthly_timeline.toPandas()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("/tmp/plots", exist_ok=True)

print("timeline_pd columns:", list(timeline_pd.columns))

dfp = timeline_pd.copy()

# 1) Identify the count column
count_col = None
for c in ["n_reviews", "count", "cnt", "reviews", "n"]:
    if c in dfp.columns:
        count_col = c
        break
if count_col is None:
    for c in dfp.columns:
        if "count" in c.lower() or "review" in c.lower():
            count_col = c
            break
if count_col is None:
    raise ValueError("Could not find a review-count column in timeline_pd.")

# 2) Build x-axis in a robust way
x = None

# Case A: there's a single monthly date column (common after date_trunc)
date_like_cols = [c for c in dfp.columns if "month" in c.lower() or "date" in c.lower() or "time" in c.lower()]
for c in date_like_cols:
    try:
        dt = pd.to_datetime(dfp[c], errors="coerce")
        if dt.notna().sum() > 0:
            dfp["_xdate"] = dt
            x = dfp["_xdate"]
            break
    except Exception:
        pass

# Case B: separate year + month integer columns
if x is None and ("year" in dfp.columns):
    if "month" in dfp.columns:
        dfp["_xdate"] = pd.to_datetime(
            dfp["year"].astype(int).astype(str) + "-" + dfp["month"].astype(int).astype(str) + "-01",
            errors="coerce"
        )
        x = dfp["_xdate"]
    else:
        # Case C: yearly only
        dfp["_xyear"] = dfp["year"].astype(int)
        x = dfp["_xyear"]

# 3) Sort + plot
if "_xdate" in dfp.columns:
    dfp = dfp.sort_values("_xdate")
    x_plot = dfp["_xdate"]
elif "_xyear" in dfp.columns:
    dfp = dfp.sort_values("_xyear")
    x_plot = dfp["_xyear"]
else:
    raise ValueError("Could not construct x-axis (no usable date/year columns found).")

y_plot = dfp[count_col]

fig = plt.figure(figsize=(10,4))
plt.plot(x_plot, y_plot)
plt.title("Amazon Review Volume Over Time")
plt.xlabel("Time")
plt.ylabel("Number of Reviews")
plt.tight_layout()

out_path = "/tmp/plots/timeline.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", out_path)


In [ ]:
from pyspark.sql import functions as F

timeline_df = (
    df2
    .withColumn("year", F.year(F.col("review_date_fixed")))
    .groupBy("year")
    .count()
    .orderBy("year")
)


In [ ]:
timeline_pd = (
    timeline_df
    .toPandas()
    .sort_values("year")
)

local_path = "/tmp/timeline_yearly.png"

plt.figure(figsize=(10,4))
plt.plot(timeline_pd["year"], timeline_pd["count"])
plt.title("Amazon Review Volume Over Time (Yearly)")
plt.xlabel("Year")
plt.ylabel("Number of Reviews")
plt.tight_layout()

# plt.savefig(local_path, dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from pyspark.sql.functions import count

top_products = (
    df.groupBy("parent_asin")
      .agg(count("*").alias("n_reviews"))
      .orderBy(col("n_reviews").desc())
      .limit(10)
)

top_products.show()

In [ ]:
top_products_pd = top_products.toPandas()


In [ ]:
top_pd = top_products.toPandas()   # rename if your variable differs

print("top_pd columns:", list(top_pd.columns))

# find product id column
pcol = None
for c in ["parent_asin", "asin", "product_id"]:
    if c in top_pd.columns:
        pcol = c
        break
if pcol is None:
    pcol = top_pd.columns[0]

# find count column
ycol = None
for c in ["n_reviews", "count", "cnt", "n"]:
    if c in top_pd.columns:
        ycol = c
        break
if ycol is None:
    for c in top_pd.columns:
        if "count" in c.lower() or "review" in c.lower():
            ycol = c
            break
if ycol is None:
    raise ValueError("No count column found in top_pd.")

top_pd = top_pd.sort_values(ycol, ascending=False).head(10).sort_values(ycol)

plt.figure(figsize=(10,4))
plt.barh(top_pd[pcol], top_pd[ycol])
plt.title("Top 10 Products by Review Count")
plt.xlabel("Number of Reviews")
plt.ylabel(pcol)
plt.tight_layout()
plt.show()

In [ ]:
top_asins = [row["parent_asin"] for row in top_products.collect()]

top_products_timeline = (
    df.filter(col("parent_asin").isin(top_asins))
      .withColumn("year", year("review_date"))
      .withColumn("month", month("review_date"))
      .groupBy("parent_asin", "year", "month")
      .agg(count("*").alias("n_reviews"))
      .orderBy("parent_asin", "year", "month")
)

top_products_timeline.show(10)


In [ ]:
category_counts = (
    df.groupBy("main_category_clean")
      .agg(count("*").alias("n_reviews"))
      .orderBy(col("n_reviews").desc())
)

category_counts.show(10)


In [ ]:
category_counts_pd = category_counts.toPandas()


In [ ]:
cat_pd = category_counts.toPandas()

print("cat_pd columns:", list(cat_pd.columns))

# find category column
ccol = None
for c in ["main_category_clean", "main_category", "category"]:
    if c in cat_pd.columns:
        ccol = c
        break
if ccol is None:
    ccol = cat_pd.columns[0]

# find count column
ycol = None
for c in ["n_reviews", "count", "cnt", "n"]:
    if c in cat_pd.columns:
        ycol = c
        break
if ycol is None:
    for c in cat_pd.columns:
        if "count" in c.lower() or "review" in c.lower():
            ycol = c
            break
if ycol is None:
    raise ValueError("No count column found in cat_pd.")

top = (
    cat_pd
    .dropna(subset=["main_category_clean"])
    .sort_values("n_reviews", ascending=False)
    .head(10)
    .copy()
)

top["main_category_clean"] = top["main_category_clean"].astype(str)

plt.figure(figsize=(10,4))
plt.barh(top["main_category_clean"][::-1], top["n_reviews"][::-1])
plt.title("Top Categories by Review Volume")
plt.xlabel("Number of Reviews")
plt.ylabel("Category")
plt.tight_layout()
plt.show()

In [ ]:
category_seasonality = (
    df
    .withColumn("month", month("review_date"))
    .groupBy("main_category_clean", "month")
    .agg(count("*").alias("n_reviews"))
)

category_seasonality.show(10)


In [ ]:
from pyspark.sql.functions import avg

product_rating_stats = (
    df.groupBy("parent_asin")
      .agg(
          count("*").alias("n_reviews"),
          avg("rating").alias("avg_rating")
      )
      .filter(col("n_reviews") >= 50)
)

product_rating_stats.show(10)
